# 03 — Feature Engineering (VacFlow)

สร้างฟีเจอร์อนุกรมเวลาต่อ (สาขา × ผลิตภัณฑ์) สำหรับโมเดลพยากรณ์ดีมานด์ 
รวมถึงค่า **Simple Moving Average** และ **Exponential Smoothing** ตามสมการใน Proposal §4.3

In [17]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
CLEAN = ROOT / 'data' / 'vaccine' / 'clean'
FEAT = ROOT / 'data' / 'vaccine' / 'features'; FEAT.mkdir(parents=True, exist_ok=True)
demand = pd.read_csv(CLEAN / 'demand_daily.csv', parse_dates=['date'])
demand = demand.sort_values(['hospital_id', 'product_id', 'date']).reset_index(drop=True)

In [18]:
# ════════════════════════════════════════════════════════════════════
# 0) ทะเบียน metadata รพ. 13 แห่ง (province / region / ประเภท / ประชากรในความดูแล)
#    — เป็น "ฐาน" ของทุก exogenous join + ใช้ทำ stratified validation
#    หมายเหตุ: ชื่อ/พิกัด รพ. อยู่ใน hospital_master.csv ส่วน metadata เชิงพื้นที่
#              (ภาค/ประเภท/catchment) ใส่ไว้ที่นี่เพราะ HOSxP ไม่ได้ให้มา
# ════════════════════════════════════════════════════════════════════
HOSP_META = pd.DataFrame([
    # hospital_id, province, region, hosp_type, catchment_pop (ประชากรในความดูแลโดยประมาณ)
    ("HOSP_001", "ตาก",           "NORTH",     "REGIONAL",    250_000),
    ("HOSP_002", "นครราชสีมา",     "NORTHEAST", "COMMUNITY",    70_000),
    ("HOSP_003", "อุดรธานี",       "NORTHEAST", "REGIONAL",    350_000),
    ("HOSP_004", "อุดรธานี",       "NORTHEAST", "COMMUNITY",    90_000),
    ("HOSP_005", "อุดรธานี",       "NORTHEAST", "COMMUNITY",    80_000),
    ("HOSP_006", "ระยอง",          "EAST",      "COMMUNITY",    90_000),
    ("HOSP_007", "กรุงเทพมหานคร",  "BKK",       "UNIVERSITY",  900_000),
    ("HOSP_008", "กรุงเทพมหานคร",  "BKK",       "REGIONAL",    600_000),
    ("HOSP_009", "กรุงเทพมหานคร",  "BKK",       "UNIVERSITY",  800_000),
    ("HOSP_010", "ปทุมธานี",       "CENTRAL",   "UNIVERSITY",  300_000),
    ("HOSP_011", "สมุทรปราการ",    "CENTRAL",   "GENERAL",     400_000),
    ("HOSP_012", "นนทบุรี",        "CENTRAL",   "SPECIALIZED",  60_000),
    ("HOSP_013", "ชลบุรี",         "EAST",      "REGIONAL",    450_000),
], columns=["hospital_id", "province", "region", "hosp_type", "catchment_pop"])

# สัดส่วนโครงสร้างอายุโดยประมาณรายภาค (เด็ก <5 ปี = กลุ่ม EPI · ผู้สูงอายุ 60+ = ไข้หวัดใหญ่)
AGE_FRAC = {  # region -> (frac_u5, frac_60p)
    "NORTH":     (0.052, 0.20), "NORTHEAST": (0.058, 0.18), "EAST": (0.055, 0.15),
    "CENTRAL":   (0.050, 0.17), "BKK":       (0.048, 0.19),
}

# โฟลเดอร์สำหรับ "ไฟล์จริง" — วาง WorldPop / รง.506 / HDC / TMD ที่นี่แล้ว loader จะ override proxy
EXT = ROOT / "data" / "external"

# ── Calibration: ดึง "ระดับ + ฤดูกาล" ของดีมานด์สังเคราะห์ให้เข้าใกล้บริการจริงของ HDC ──
#    ลด distribution shift ก่อน train  ·  ไฟล์จริง:
#    data/external/hdc/vaccine_service_monthly.csv  (province, year_month=YYYY-MM, doses)
def calibrate_demand(demand):
    f = EXT / "hdc" / "vaccine_service_monthly.csv"
    if not f.exists():
        print("[calib] ไม่พบ HDC reference -> ข้าม (ดีมานด์คงเดิม). วางไฟล์ที่:", f)
        return demand
    ref = pd.read_csv(f).set_index(["province", "year_month"])["doses"]
    prov = HOSP_META.set_index("hospital_id")["province"]
    d = demand.copy()
    d["province"] = d["hospital_id"].map(prov)
    d["year_month"] = d["date"].dt.strftime("%Y-%m")
    syn = d.groupby(["province", "year_month"])["demand"].sum().rename("syn")
    scale = (ref / syn).replace([np.inf, -np.inf], np.nan).dropna()
    d = d.join(scale.rename("scale"), on=["province", "year_month"])
    d["scale"] = d["scale"].fillna(1.0).clip(0.2, 5.0)   # กันสเกลสุดโต่งจากเดือนที่ข้อมูลน้อย
    d["demand"] = (d["demand"] * d["scale"]).round().astype(int)
    print(f"[calib] ปรับดีมานด์ตาม HDC แล้ว: {scale.notna().sum()} (province, month)")
    return d[["date", "hospital_id", "product_id", "demand"]]

demand = calibrate_demand(demand)
print("demand (หลัง calibrate):", demand.shape)

[calib] ไม่พบ HDC reference -> ข้าม (ดีมานด์คงเดิม). วางไฟล์ที่: c:\Work\LogisticsInnovationHackathon2026\VacFlow\data\external\hdc\vaccine_service_monthly.csv
demand (หลัง calibrate): (353340, 4)


## 1) ฟีเจอร์เวลา + lag + rolling (SMA)

`F(t+1) = (Σ D(t-i+1)) / n` — ค่าเฉลี่ยเคลื่อนที่ใช้เป็นพยากรณ์สภาวะปกติ

In [19]:
def add_features(g, sma_window=7):
    g = g.copy()
    g['dow'] = g['date'].dt.weekday
    g['is_weekend'] = (g['dow'] >= 5).astype(int)
    for lag in (1, 7, 14):
        g[f'lag_{lag}'] = g['demand'].shift(lag)
    g['sma_7'] = g['demand'].shift(1).rolling(7).mean()        # SMA (พยากรณ์ปกติ)
    g['sma_14'] = g['demand'].shift(1).rolling(14).mean()
    g['roll_std_7'] = g['demand'].shift(1).rolling(7).std()    # ความผันผวน
    return g

feat = demand.groupby(['hospital_id', 'product_id'], group_keys=False).apply(add_features)

C:\Users\jetan\AppData\Local\Temp\ipykernel_53952\4099612143.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feat = demand.groupby(['hospital_id', 'product_id'], group_keys=False).apply(add_features)


## 2) Exponential Smoothing (สภาวะวิกฤต)

`F(t+1) = α·D(t) + (1-α)·F(t)` — ให้น้ำหนักข้อมูลล่าสุดมากกว่า เหมาะกับดีมานด์ที่ผันผวน

In [20]:
def exp_smoothing(series, alpha=0.4):
    f = np.zeros(len(series)); f[0] = series.iloc[0]
    for t in range(1, len(series)):
        f[t] = alpha * series.iloc[t - 1] + (1 - alpha) * f[t - 1]
    return f

feat['es_0.4'] = (feat.groupby(['hospital_id', 'product_id'])['demand']
                      .transform(lambda s: exp_smoothing(s, 0.4)))
feat = feat.dropna().reset_index(drop=True)
print('features:', feat.shape)
feat.head()

features: (325858, 13)


,date,hospital_id,product_id,demand,dow,is_weekend,lag_1,lag_7,lag_14,sma_7,sma_14,roll_std_7,es_0.4
0,2026-01-11,HOSP_001,VAX_J07AE01_051,14,6,1,13.0,9.0,4.0,18.142857,15.357143,5.871643,16.364677
1,2026-01-12,HOSP_001,VAX_J07AE01_051,17,0,0,14.0,23.0,16.0,18.857143,16.071429,4.775932,15.418806
2,2026-01-13,HOSP_001,VAX_J07AE01_051,16,1,0,17.0,21.0,7.0,18.000000,16.142857,4.434712,16.051284
3,2026-01-14,HOSP_001,VAX_J07AE01_051,13,2,0,16.0,25.0,15.0,17.285714,16.785714,4.270608,16.030770
4,2026-01-15,HOSP_001,VAX_J07AE01_051,12,3,0,13.0,21.0,9.0,15.571429,16.642857,2.819997,14.818462


## 4) Exogenous features — ประชากร/อายุ (WorldPop) · สัญญาณโรค (รง.506) · อากาศ (TMD)

โมเดลเดิมเป็น **univariate** (มีแค่ lag/SMA/ES ของดีมานด์ตัวเอง) จึง bias ตามพื้นที่/ฤดูกาล
ที่ข้อมูลสังเคราะห์กำหนดไว้ ส่วนนี้เติม **ตัวแปรภายนอกจากโลกจริง** เพื่อให้โมเดลอธิบายดีมานด์
ด้วยปัจจัยจริง (จำนวนประชากร, โครงสร้างอายุ, การระบาดไข้หวัดใหญ่, ฤดูฝน)

> ทุก loader ออกแบบเป็น **drop-in**: ถ้ามีไฟล์จริงใน `data/external/...` จะใช้ของจริง
> ถ้าไม่มีจะ fallback เป็น proxy ที่อิงพื้นที่จริงของ รพ. 13 แห่ง (รันได้เลยไม่ต้องรอ download)

| feature | แหล่งจริง | แก้ bias |
| --- | --- | --- |
| `pop_total`,`frac_u5`,`frac_60p` | WorldPop (age/sex 100m) | geographic/population |
| `ili_rate` | รง.506 / DDS (DDC) | seasonality + outbreak |
| `rainy_season`,`temp_c` | กรมอุตุนิยมวิทยา (TMD) | seasonality |
| `region`,`hosp_type`,`small_hosp` | metadata | stratified validation + reporting bias |

In [21]:
# ── loader: ประชากร (WorldPop) ──────────────────────────────────────
#    ไฟล์จริง: data/external/worldpop/hospital_population.csv
#      (hospital_id, pop_total, pop_u5, pop_60p) — เตรียมจาก zonal-stat ของ raster 100m
def load_population():
    f = EXT / "worldpop" / "hospital_population.csv"
    if f.exists():
        print(f"[pop] โหลดไฟล์จริง {f.name}")
        return pd.read_csv(f)
    print("[pop] ไม่พบไฟล์จริง -> proxy (catchment_pop x โครงสร้างอายุรายภาค)")
    rows = []
    for r in HOSP_META.itertuples(index=False):
        fu5, f60 = AGE_FRAC[r.region]
        rows.append({"hospital_id": r.hospital_id, "pop_total": r.catchment_pop,
                     "pop_u5": int(r.catchment_pop * fu5),
                     "pop_60p": int(r.catchment_pop * f60)})
    return pd.DataFrame(rows)

# ── loader: ไข้หวัดใหญ่รายสัปดาห์ (รง.506/DDS) ──────────────────────
#    ไฟล์จริง: data/external/ddc_506/influenza_weekly_province.csv
#      (province, iso_year, iso_week, cases) จาก opendata.ddc.moph.go.th
def _seasonal_ili(week, region):
    """ILI ต่อแสน (proxy) — พีคหลักหน้าฝน (สัปดาห์ ~34) + พีครองต้นปี (~5)."""
    main = np.exp(-((week - 34) ** 2) / (2 * 6.0 ** 2))
    minor = 0.55 * np.exp(-((week - 5) ** 2) / (2 * 4.0 ** 2))
    base = {"NORTH": 1.05, "NORTHEAST": 1.10, "EAST": 0.95,
            "CENTRAL": 1.00, "BKK": 0.90}[region]
    return float(40 * base * (main + minor) + 6)

def load_influenza(date_prov):
    """คืน (date, province, ili_rate) ให้ทุกคู่วัน-จังหวัดที่โมเดลใช้."""
    cal = date_prov.copy()
    iso = cal["date"].dt.isocalendar()
    cal["iso_year"], cal["iso_week"] = iso["year"].astype(int), iso["week"].astype(int)
    f = EXT / "ddc_506" / "influenza_weekly_province.csv"
    if f.exists():
        print(f"[506] โหลดไฟล์จริง {f.name}")
        flu = pd.read_csv(f).merge(
            HOSP_META[["province", "catchment_pop"]].drop_duplicates("province"),
            on="province", how="left")
        flu["ili_rate"] = flu["cases"] / flu["catchment_pop"] * 1e5
        out = cal.merge(flu[["province", "iso_year", "iso_week", "ili_rate"]],
                        on=["province", "iso_year", "iso_week"], how="left")
    else:
        print("[506] ไม่พบไฟล์จริง -> synth ฤดูกาลไข้หวัดใหญ่ (พีคหน้าฝน)")
        reg = HOSP_META.set_index("province")["region"].to_dict()
        out = cal.copy()
        out["ili_rate"] = [_seasonal_ili(w, reg.get(p, "CENTRAL"))
                           for w, p in zip(out["iso_week"], out["province"])]
    return out[["date", "province", "ili_rate"]]

# ── loader: อากาศ (TMD) — fallback เป็นค่าเฉลี่ยรายเดือนของไทย ──────
def add_climate(df):
    m = df["date"].dt.month
    df["rainy_season"] = m.between(5, 10).astype(int)           # พ.ค.–ต.ค.
    temp_by_month = {1:27,2:29,3:31,4:32,5:31,6:30,7:29,8:29,9:29,10:29,11:28,12:27}
    df["temp_c"] = m.map(temp_by_month)
    return df

print("loaders พร้อม:", [f.__name__ for f in (load_population, load_influenza, add_climate)])

loaders พร้อม: ['load_population', 'load_influenza', 'add_climate']


In [22]:
# ── merge exogenous เข้า feat ──────────────────────────────────────
feat = feat.merge(HOSP_META, on="hospital_id", how="left")

pop = load_population()
pop["log_pop"] = np.log1p(pop["pop_total"])
pop["frac_u5"] = pop["pop_u5"] / pop["pop_total"]
pop["frac_60p"] = pop["pop_60p"] / pop["pop_total"]
feat = feat.merge(pop[["hospital_id", "pop_total", "log_pop", "frac_u5", "frac_60p"]],
                  on="hospital_id", how="left")

flu = load_influenza(feat[["date", "province"]].drop_duplicates())
feat = feat.merge(flu, on=["date", "province"], how="left")
feat = add_climate(feat)

# ════════════════════════════════════════════════════════════════════
# 5) Reporting-bias guard — "0 = ไม่รายงาน" ไม่ใช่ "0 = ไม่มีความต้องการ"
#    รพ.เล็ก (ชุมชน/เฉพาะทาง) มักบันทึกไม่ครบ → อย่าให้โมเดลเรียนว่าดีมานด์=0 จริง
# ════════════════════════════════════════════════════════════════════
SMALL_TYPES = {"COMMUNITY", "SPECIALIZED"}
feat["small_hosp"] = feat["hosp_type"].isin(SMALL_TYPES).astype(int)
# flag ไว้ให้โมเดล/ผู้ใช้รู้ (ไม่บิดค่า demand จริง) — กันสรุปผิดว่า รพ.เล็กไม่ต้องการวัคซีน
feat["zero_demand_small"] = ((feat["demand"] == 0) & (feat["small_hosp"] == 1)).astype(int)
# ili_rate ที่ไม่มีรายงาน (จาก 506 จริง) = ข้อมูลขาด ไม่ใช่ความเสี่ยง 0 → เติมมัธยฐานรายภาค
feat["ili_reported"] = feat["ili_rate"].notna().astype(int)
feat["ili_rate"] = feat.groupby("region")["ili_rate"].transform(lambda s: s.fillna(s.median()))
feat["ili_rate"] = feat["ili_rate"].fillna(feat["ili_rate"].median())  # กันภาคที่ว่างทั้งภาค

EXOG = ["log_pop", "frac_u5", "frac_60p", "ili_rate", "rainy_season", "temp_c",
        "small_hosp", "zero_demand_small", "ili_reported"]
print("exogenous ที่เพิ่ม:", EXOG)
print("NaN รวมในคอลัมน์ใหม่:", int(feat[EXOG].isna().sum().sum()))
print("ili_rate เฉลี่ยรายเดือน (เช็คฤดูกาล):",
      feat.groupby(feat["date"].dt.month)["ili_rate"].mean().round(1).to_dict())
feat[["date", "hospital_id", "region", "hosp_type", "demand"] + EXOG].head()

[pop] ไม่พบไฟล์จริง -> proxy (catchment_pop x โครงสร้างอายุรายภาค)
[506] ไม่พบไฟล์จริง -> synth ฤดูกาลไข้หวัดใหญ่ (พีคหน้าฝน)
exogenous ที่เพิ่ม: ['log_pop', 'frac_u5', 'frac_60p', 'ili_rate', 'rainy_season', 'temp_c', 'small_hosp', 'zero_demand_small', 'ili_reported']
NaN รวมในคอลัมน์ใหม่: 0
ili_rate เฉลี่ยรายเดือน (เช็คฤดูกาล): {1: 26.7, 2: 24.1, 3: 12.3, 4: 7.2, 5: 9.2, 6: 17.2}


,date,hospital_id,region,hosp_type,demand,log_pop,frac_u5,frac_60p,ili_rate,rainy_season,temp_c,small_hosp,zero_demand_small,ili_reported
0,2026-01-11,HOSP_001,NORTH,REGIONAL,14,12.42922,0.052,0.2,23.436823,0,27,0,0,1
1,2026-01-12,HOSP_001,NORTH,REGIONAL,17,12.42922,0.052,0.2,26.385746,0,27,0,0,1
2,2026-01-13,HOSP_001,NORTH,REGIONAL,16,12.42922,0.052,0.2,26.385746,0,27,0,0,1
3,2026-01-14,HOSP_001,NORTH,REGIONAL,13,12.42922,0.052,0.2,26.385746,0,27,0,0,1
4,2026-01-15,HOSP_001,NORTH,REGIONAL,12,12.42922,0.052,0.2,26.385746,0,27,0,0,1


## 3) บันทึกชุดฟีเจอร์

In [23]:
feat.to_csv(FEAT / 'demand_features.csv', index=False, encoding='utf-8-sig')
print('[saved]', (FEAT / 'demand_features.csv').resolve())
print('rows:', len(feat), '| columns:', len(feat.columns))
print('exogenous:', EXOG)
# manifest รายชื่อฟีเจอร์ exogenous — ให้ notebook 04/05 อ่านไปใช้ได้โดยไม่ hardcode
(FEAT / 'exogenous_features.txt').write_text('\n'.join(EXOG), encoding='utf-8')
print('[saved]', (FEAT / 'exogenous_features.txt').name)

[saved] C:\Work\LogisticsInnovationHackathon2026\VacFlow\data\vaccine\features\demand_features.csv
rows: 325858 | columns: 27
exogenous: ['log_pop', 'frac_u5', 'frac_60p', 'ili_rate', 'rainy_season', 'temp_c', 'small_hosp', 'zero_demand_small', 'ili_reported']
[saved] exogenous_features.txt


## 6) Stratified validation helper — จับ bias เชิงพื้นที่ที่ค่า error รวมซ่อนไว้

ค่า RMSE รวมทั้งเครือข่ายอาจดูดี แต่ซ่อนการพยากรณ์พลาดหนักใน รพ.ชุมชน/บางภาค
ฟังก์ชัน `stratified_scores` แตกคะแนนตาม `hosp_type` และ `region` — notebook 04/05 เรียกใช้ซ้ำได้
(ตรงนี้เดโมด้วย baseline SMA-7 เพื่อยืนยันว่า error ไม่กระจายเท่ากันทุกกลุ่ม)

In [24]:
def stratified_scores(df, actual="demand", pred="sma_7", by=("hosp_type", "region")):
    """RMSE/MAE/n แยกตามกลุ่ม (เช่น ประเภท รพ., ภาค) — ใช้จับ bias เชิงพื้นที่."""
    out = {}
    for col in by:
        g = df.dropna(subset=[actual, pred]).groupby(col)
        out[col] = (g.apply(lambda x: pd.Series({
            "n": len(x),
            "MAE": (x[actual] - x[pred]).abs().mean(),
            "RMSE": np.sqrt(((x[actual] - x[pred]) ** 2).mean()),
        })).round(2))
    return out

for col, tbl in stratified_scores(feat).items():
    print(f"\n── error แยกตาม {col} (baseline SMA-7) ──")
    print(tbl)


── error แยกตาม hosp_type (baseline SMA-7) ──
                    n   MAE   RMSE
hosp_type                         
COMMUNITY    100264.0  6.16   9.05
GENERAL       25066.0  5.58   7.98
REGIONAL     100264.0  6.55   9.74
SPECIALIZED   25066.0  7.62  11.15
UNIVERSITY    75198.0  6.64   9.64

── error แยกตาม region (baseline SMA-7) ──
                  n   MAE   RMSE
region                          
BKK         75198.0  7.55  11.00
CENTRAL     75198.0  6.17   9.04
EAST        50132.0  6.72   9.95
NORTH       25066.0  5.02   7.24
NORTHEAST  100264.0  6.09   8.88


C:\Users\jetan\AppData\Local\Temp\ipykernel_53952\2204269933.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out[col] = (g.apply(lambda x: pd.Series({
C:\Users\jetan\AppData\Local\Temp\ipykernel_53952\2204269933.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out[col] = (g.apply(lambda x: pd.Series({
